# SRT-M15：手机训练自主底层模型（从零初始化）

本 Notebook **已打包所有源代码**，无需另外上传 ZIP。使用 Google Colab 执行，每个模块从随机权重开始。

包含：手写注意力 Transformer、三推理分支、双向动态通信、三尺度内部记忆、外部 JSON 记忆、基于可靠标签的在线更新，以及字符级文本生成。**这是研究原型，不是已训练好的聊天大模型，也不是经过证明的原创架构。**


## 第 1 步：解压内嵌的源代码
下面第一段代码将代码展开到当前 Colab 运行环境。

In [ ]:
import io, zipfile, base64, os
SOURCE_ZIP_BASE64 = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_ZIP_BASE64))) as z:
    z.extractall("/content")
os.chdir("/content/SRT_M15_Starter")
print("工作目录:", os.getcwd())
print("发现的源码:", sorted(os.listdir("srt_m15")))


## 第 2 步：测试网络结构
单元测试验证梯度、因果注意力不泄露未来信息、独立网络可关闭通信，以及在线更新等。

In [ ]:
!python -m unittest discover -s tests -v


## 第 3 步：从零训练三个协作神经网络
小任务：网络 A 看到第一个数字，B 看到第二个数字，C 看不到两个数字；**只有 C 的答案用来训练**。如果 C 学会加法，就必须通过 A/B 的内部通信获得信息。

In [ ]:
!python train.py --steps 180 --task distributed --device auto --log-every 45


## 第 4 步：测试推理、路由、记忆、在线更新与消融

In [ ]:
!python demo.py


## 第 5 步（可选）：从零训练字符级语言生成
这是另一条模型训练路径，只学习小型示范语料中的字符规律。它不会成为 ChatGPT 级别的模型。

In [ ]:
!python train_lm.py --steps 65 --device auto --log-every 20


In [ ]:
!python generate.py --prompt "你好" --new-tokens 25


## 第 6 步：下载模型权重和训练记录
训练完成后，将权重文件保存到手机或云盘；Colab 运行环境可能被清理。

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
output = Path("/content/SRT_M15_Results.zip")
with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["srt_m15_toy.pt", "srt_m15_lm.pt", "metrics.json", "knowledge.json"]:
        path = Path("checkpoints") / name
        if path.exists():
            z.write(path, arcname=path.as_posix())
print("已收集：", output)
files.download(str(output))
